# Task 2 hyperparameter sweeps

One cell per model. `run_grid_search(task, model, parameter_grid)`:

- **a list** in `parameter_grid` = values to try (every combination of all lists is run)
- **a single value** (or a one-element list) = constant in every run
- **not mentioned** = the value from `submission/lib/config.py` (Task 2 section)
- **nested settings** as `"outer.inner"`, e.g. `"temporal_biases.drift_regularization"`

Every run trains on the training split and is scored on the validation split: **pair accuracy (higher is better)**.
No refit on all data, nothing is written to `trained_models/` -> the submitted models stay untouched.

**Result:** `output/task2/<model>_<start time>.json` with the best combination and its full parameter set
(`best.parameters`, can be copied into `lib/config.py`), the swept and constant values and the score of every run.
The file is rewritten after every run: interrupting a cell keeps the finished runs.

**Optional:** `rerun_finalists_on_other_splits=True` re-trains the best `n_finalists` (default 3) combinations on the validation splits with seeds 1 and 2; the winner is then the best mean over the three splits.
`show_training_log=True` prints the training's own log lines of every run.

The example ranges are around the current values in `lib/config.py`; the first printed run time shows how long a sweep takes.
`popularity` has no hyperparameters and is not sweepable.
EASE needs ~21 GB RAM per fit (all 51k books) -> do not run two EASE sweeps (Task 2 / 4 / 5) at the same time.

In [ ]:
%load_ext autoreload
%autoreload 2

# grid_search.py lives in this folder and imports assignment1.py / lib/ from ../submission
import torch

from grid_search import print_sweep_overview, run_grid_search

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## Item cosine similarity (CPU)

In [ ]:
item_cosine_sweep = run_grid_search("task2", "item_cosine", {
    "n_neighbors": [50, 100, 200],
    "similarity_shrinkage": [10.0, 20.0, 50.0],
})

## EASE (CPU, in RAM, several minutes per run)

In [ ]:
ease_sweep = run_grid_search("task2", "ease", {
    "regularization": [250.0, 500.0, 1000.0],
})

## Mult-VAE (GPU, early stopping on the pair accuracy)

In [ ]:
mult_vae_sweep = run_grid_search("task2", "mult_vae", {
    "input_dropout": [0.3, 0.5],
    "kl_weight_max": [0.1, 0.2, 0.3],
}, rerun_finalists_on_other_splits=False, n_finalists=2)

## Overview

In [ ]:
# one line per sweep file in output/task2 (also works right after a kernel restart)
print_sweep_overview("task2")